# Week 10: Introduction to LangChain

This notebook introduces LangChain concepts and demonstrates how to build LLM applications.

## Topics Covered

1. LLM Setup and Basic Chains
2. Document Loading
3. Text Splitting
4. Vector Stores and Embeddings
5. Retrieval QA Chain
6. Building a Complete Pipeline

In [ ]:
# Install required packages (if needed)
# !pip install langchain langchain-openai langchain-community
# !pip install chromadb pypdf python-dotenv

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API key
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  Please set OPENAI_API_KEY in your .env file")
else:
    print("✅ API key loaded successfully")

## 1. LLM Setup and Basic Chains

Let's start with the simplest LangChain component - the LLM.

In [ ]:
from langchain_openai import OpenAI, ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Initialize LLM
llm = OpenAI(temperature=0.7)
chat_model = ChatOpenAI(temperature=0.7, model="gpt-3.5-turbo")

# Simple prompt template
template = """
You are a helpful assistant. Answer the following question:

Question: {question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)

# Create chain
chain = LLMChain(llm=llm, prompt=prompt)

# Run chain
result = chain.predict(question="What is machine learning?")
print(result)

In [ ]:
# Using the newer LCEL (LangChain Expression Language) syntax
from langchain_core.output_parsers import StrOutputParser

# Modern chain composition
modern_chain = (
    prompt 
    | chat_model 
    | StrOutputParser()
)

result = modern_chain.invoke({"question": "Explain neural networks in simple terms"})
print(result)

## 2. Document Loading

LangChain supports loading documents from various sources.

In [ ]:
from langchain.document_loaders import TextLoader, PyPDFLoader

# Create a sample text file
sample_text = """
LangChain is a framework for developing applications powered by language models.

It enables applications that:
- Are context-aware: connect a language model to sources of context
- Reason: rely on a language model to reason about the context

LangChain provides:
1. Components: building blocks for working with language models
2. Chains: structured compositions of components
3. Agents: use LLMs to choose actions
"""

# Save sample
with open("/tmp/sample.txt", "w") as f:
    f.write(sample_text)

# Load with TextLoader
loader = TextLoader("/tmp/sample.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
print(f"Content preview: {documents[0].page_content[:200]}...")
print(f"Metadata: {documents[0].metadata}")

In [ ]:
# Available document loaders
loaders_info = {
    "TextLoader": "Plain text files",
    "PyPDFLoader": "PDF documents",
    "CSVLoader": "CSV files",
    "JSONLoader": "JSON documents",
    "UnstructuredHTMLLoader": "HTML pages",
    "DirectoryLoader": "All files in a directory"
}

print("Available Document Loaders:")
for loader, use_case in loaders_info.items():
    print(f"  • {loader}: {use_case}")

## 3. Text Splitting

Documents are often too long for the model's context window. We need to split them into chunks.

In [ ]:
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

# Sample long text
long_text = """
Artificial Intelligence (AI) is intelligence demonstrated by machines,
as opposed to the natural intelligence displayed by animals including humans.

AI research has been defined as the field of study of intelligent agents,
which refers to any system that perceives its environment and takes actions
that maximize its chance of achieving its goals.

The term "artificial intelligence" had previously been used to describe machines
that mimic and display "human" cognitive skills that are associated with the
human mind, such as "learning" and "problem-solving". This definition has since
been rejected by major AI researchers who now describe AI in terms of rationality
and acting rationally, which does not limit how intelligence can be articulated.
""" * 5  # Repeat to make it longer

print(f"Original text length: {len(long_text)} characters")

In [ ]:
# Recursive Character Text Splitter (Recommended)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

recursive_chunks = recursive_splitter.split_text(long_text)
print(f"Recursive splitter created {len(recursive_chunks)} chunks")

# Show chunk sizes
sizes = [len(chunk) for chunk in recursive_chunks]
print(f"Chunk sizes: min={min(sizes)}, max={max(sizes)}, avg={sum(sizes)//len(sizes)}")

In [ ]:
# Compare different splitters

# Character Text Splitter
char_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separator="\n"
)

char_chunks = char_splitter.split_text(long_text)
print(f"Character splitter: {len(char_chunks)} chunks")

# Visualize chunking
print("\nFirst few chunks from recursive splitter:")
for i, chunk in enumerate(recursive_chunks[:3]):
    preview = chunk.replace('\n', ' ')[:100]
    print(f"\nChunk {i+1} ({len(chunk)} chars): {preview}...")

## 4. Vector Stores and Embeddings

Convert text chunks into embeddings and store them for efficient retrieval.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.docstore.document import Document

# Initialize embeddings
embeddings = OpenAIEmbeddings()

# Convert chunks to Document objects
docs = [Document(page_content=chunk) for chunk in recursive_chunks[:10]]

# Create vector store
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="/tmp/chroma_db"
)

print(f"Vector store created with {len(docs)} documents")
print(f"Embedding dimension: 1536 (for text-embedding-ada-002)")

In [ ]:
# Test similarity search
query = "What is artificial intelligence?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
print("Top 3 most relevant chunks:")
for i, doc in enumerate(results, 1):
    preview = doc.page_content[:150].replace('\n', ' ')
    print(f"{i}. {preview}...\n")

In [ ]:
# Vector store with metadata
docs_with_meta = [
    Document(
        page_content="LangChain is a framework for LLM applications",
        metadata={"source": "intro.txt", "topic": "framework"}
    ),
    Document(
        page_content="OpenAI provides GPT models for text generation",
        metadata={"source": "models.txt", "topic": "models"}
    ),
    Document(
        page_content="Chroma is a vector database for embeddings",
        metadata={"source": "database.txt", "topic": "storage"}
    )
]

# Create new vector store
metadata_store = Chroma.from_documents(
    documents=docs_with_meta,
    embedding=embeddings
)

# Search with filter
results = metadata_store.similarity_search(
    "LLM frameworks",
    k=2,
    filter={"topic": "framework"}
)

print("Filtered results:")
for doc in results:
    print(f"  - {doc.page_content}")
    print(f"    Source: {doc.metadata['source']}")

## 5. Retrieval QA Chain

Combine retrieval with question answering.

In [ ]:
from langchain.chains import RetrievalQA

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=chat_model,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("QA Chain created successfully")
print(f"Retriever config: k=3, search_type=similarity")

In [ ]:
# Ask questions
questions = [
    "What is AI?",
    "How do AI systems make decisions?",
    "What are the goals of AI research?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print('='*60)
    
    result = qa_chain.invoke({"query": question})
    
    print(f"A: {result['result']}\n")
    print("Sources:")
    for i, doc in enumerate(result['source_documents'], 1):
        preview = doc.page_content[:100].replace('\n', ' ')
        print(f"  [{i}] {preview}...")

In [ ]:
# Custom prompt for QA
from langchain.prompts import PromptTemplate

custom_prompt = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know.
Be concise and specific in your answer.

Context:
{context}

Question: {question}

Concise Answer:
"""

PROMPT = PromptTemplate(
    template=custom_prompt,
    input_variables=["context", "question"]
)

# Create chain with custom prompt
custom_qa = RetrievalQA.from_chain_type(
    llm=chat_model,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT}
)

# Test with custom prompt
result = custom_qa.invoke({"query": "What is the definition of AI?"})
print(result['result'])

## 6. Building a Complete Pipeline

Let's put it all together into a complete document Q&A system.

In [ ]:
class DocumentQA:
    """Complete document Q&A pipeline."""
    
    def __init__(self, chunk_size=1000, chunk_overlap=200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.embeddings = OpenAIEmbeddings()
        self.llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
        self.vectorstore = None
        self.qa_chain = None
    
    def load_and_process(self, texts):
        """Load texts and create vector store."""
        # Split texts
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap
        )
        
        docs = [Document(page_content=t) for t in texts]
        chunks = splitter.split_documents(docs)
        
        # Create vector store
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings
        )
        
        # Create QA chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            retriever=self.vectorstore.as_retriever(k=4)
        )
        
        return len(chunks)
    
    def ask(self, question):
        """Ask a question about the documents."""
        if not self.qa_chain:
            return "Please load documents first."
        return self.qa_chain.invoke({"query": question})['result']

# Usage
qa_system = DocumentQA(chunk_size=500, chunk_overlap=50)

sample_docs = [
    "Python is a high-level programming language. It was created by Guido van Rossum.",
    "JavaScript is used for web development. It runs in browsers and on servers via Node.js.",
    "Machine learning is a subset of AI. It uses algorithms to learn from data."
]

num_chunks = qa_system.load_and_process(sample_docs)
print(f"Processed {num_chunks} chunks from {len(sample_docs)} documents\n")

# Ask questions
questions = [
    "Who created Python?",
    "What is JavaScript used for?",
    "How does machine learning work?"
]

for q in questions:
    answer = qa_system.ask(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## Summary

### Key Concepts

1. **Document Loaders**: Ingest data from various sources
2. **Text Splitters**: Break documents into manageable chunks
3. **Embeddings**: Convert text to vector representations
4. **Vector Stores**: Efficient storage and retrieval of embeddings
5. **Retrievers**: Find relevant context for queries
6. **Chains**: Compose components into pipelines

### Best Practices

- Use `RecursiveCharacterTextSplitter` for general text
- Choose chunk size based on your content and model context window
- Include 10-20% overlap between chunks
- Use metadata to filter and organize documents
- Test different retrieval configurations (k, search_type)

### Next Steps

- Week 11: Advanced RAG techniques
- Memory and conversational chains
- Agents and tool use
- Evaluation and monitoring